**This code is to use shape descriptors for classification using multi-layer perceptrons (MLPs):**

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA, FastICA, NMF, TruncatedSVD
from sklearn.random_projection import GaussianRandomProjection, SparseRandomProjection
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [2]:
# ---------------- CONFIG ----------------
MODEL = "padc_cub_dim_search"
DEVICE_ID = 0
SEED = 42
BATCH_SIZE = 512
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 1e-5
DROPOUT = 0.3
EXPLAINED_VARIANCE_THRESHOLD = 0.99

DATA_DIR = r'/home/c/choton/beemachine/datasets/Others/CUB_200_2011/'
LOG_DIR = f"./{MODEL}_logs/"
cub_feats_path = r"/home/c/choton/beemachine/codes/AG_vision_2026/4_shape_feature_analysis/CUB/shape_features_cub/cub_shape_features_updated.csv"

device = torch.device(f"cuda:{DEVICE_ID}" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

In [3]:
# ---------------- FAST IMPUTATION ----------------
def fill_by_class_mean(df, class_col="label"):
    df = df.copy()

    # Replace zeros with NaN
    df.replace(0, np.nan, inplace=True)

    # Keep only columns that are not all NaN
    df = df.dropna(axis=1, how="all")

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    # Compute class means once
    class_means = df.groupby(class_col)[numeric_cols].transform("mean")

    # Fill NaNs with class means
    df[numeric_cols] = df[numeric_cols].fillna(class_means)

    # Global fallback
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

    return df

# ---------------- DATASET ----------------
class ShapeFeatureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# The MLP classifier Network
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, num_classes, dropout=0.3):
        super().__init__()
        mid_dim = input_dim
        if (input_dim // 2) > num_classes:
            mid_dim = input_dim // 2
        self.net = nn.Sequential(
            nn.Linear(input_dim, mid_dim),
            nn.LayerNorm(mid_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mid_dim, num_classes)
        )
        # simple init
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)

def reduce_dimensions(X_train, X_val, X_test, method="PCA", n_components=100, random_state=SEED):
    """
    Apply dimensionality reduction on the dataset.
    Supported methods: "PCA", "ICA", "NMF", "TruncatedSVD", "GRP", "SRP"
    """
    if method == "PCA":
        reducer = PCA(n_components=n_components, random_state=random_state)
    elif method == "ICA":
        reducer = FastICA(n_components=n_components, random_state=random_state, max_iter=1000)
    elif method == "NMF":
        # NMF requires non-negative input
        X_train_nonneg = np.clip(X_train, 0, None)
        X_val_nonneg = np.clip(X_val, 0, None)
        X_test_nonneg = np.clip(X_test, 0, None)
        reducer = NMF(n_components=n_components, init='nndsvda', random_state=random_state, max_iter=1000)
        X_train = reducer.fit_transform(X_train_nonneg)
        X_val = reducer.transform(X_val_nonneg)
        X_test = reducer.transform(X_test_nonneg)
        return X_train, X_val, X_test, reducer
    elif method == "TruncatedSVD":
        reducer = TruncatedSVD(n_components=n_components, random_state=random_state)
    elif method == "GRP":
        reducer = GaussianRandomProjection(n_components=n_components, random_state=random_state)
    elif method == "SRP":
        reducer = SparseRandomProjection(n_components=n_components, random_state=random_state)
    else:
        raise ValueError(f"Unknown dimensionality reduction method: {method}")
    
    X_train_reduced = reducer.fit_transform(X_train)
    X_val_reduced   = reducer.transform(X_val)
    X_test_reduced  = reducer.transform(X_test)
    
    return X_train_reduced, X_val_reduced, X_test_reduced, reducer

def select_n_components_via_pca(X_train, explained_var_threshold=0.95, random_state=SEED, plot=False):
    """Use PCA to select optimal n_components based on explained variance."""
    pca_full = PCA(n_components=X_train.shape[1], random_state=random_state)
    pca_full.fit(X_train)
    cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
    n_components_opt = np.argmax(cumulative_variance >= explained_var_threshold) + 1
    print(f"Optimal n_components (PCA, {explained_var_threshold*100}% variance): {n_components_opt}")
    
    # Optional plot
    if plot:
        plt.figure(figsize=(8,5))
        plt.plot(cumulative_variance, marker='o')
        plt.axhline(y=explained_var_threshold, color='r', linestyle='--')
        plt.xlabel('Number of components')
        plt.ylabel('Cumulative explained variance')
        plt.title('PCA Explained Variance')
        plt.grid(True)
        plt.show()    
    return n_components_opt


# Training functions
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)

        preds = outputs.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    return total_loss / total, correct / total

def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)

            total_loss += loss.item() * X.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / total, correct / total

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [4]:
# 2. LOAD OFFICIAL SPLITS

images_df = pd.read_csv(os.path.join(DATA_DIR, "images.txt"), sep=" ", names=["img_id", "filepath"])
labels_df = pd.read_csv(os.path.join(DATA_DIR, "image_class_labels.txt"), sep=" ", names=["img_id", "label"])
split_df = pd.read_csv(os.path.join(DATA_DIR, "train_test_split.txt"), sep=" ", names=["img_id", "is_train"])

# Merge all
df = images_df.merge(labels_df, on="img_id")
df = df.merge(split_df, on="img_id")

# Convert labels to 0-index
df["label"] = df["label"] - 1

# =========================
# 3. CREATE 75:15:10 SPLIT
# =========================

# First separate official test
official_train = df[df["is_train"] == 1]
official_test = df[df["is_train"] == 0]

# Combine all data
full_df = pd.concat([official_train, official_test])

# 75% train, 25% temp
train_df, temp_df = train_test_split(
    full_df,
    test_size=0.25,
    stratify=full_df["label"],
    random_state=SEED
)

# Split temp into val (15%) and test (10%)
val_ratio = 0.15 / (0.15 + 0.10)

val_df, test_df = train_test_split(temp_df, test_size=(1 - val_ratio), stratify=temp_df["label"], random_state=SEED)
print(f"Train: {len(train_df)}, Val:   {len(val_df)}, Test:  {len(test_df)}")

columns_to_drop = train_df.columns
print("Current columns:", columns_to_drop)

Train: 8841, Val:   1768, Test:  1179
Current columns: Index(['img_id', 'filepath', 'label', 'is_train'], dtype='object')


In [5]:
full_df = pd.read_csv(cub_feats_path)
full_df

,image,class_id,part1_area,part1_perimeter,part1_aspect_ratio,part1_extent,part1_solidity,part1_eccentricity,part1_orientation,part1_circularity,...,area_ratio_part8_to_total,area_ratio_part9_to_total,area_ratio_part3_to_part11,area_ratio_part4_to_part8,area_ratio_part4_to_part9,area_ratio_part8_to_part10,area_ratio_part9_to_part10,area_ratio_part3_to_part8,area_ratio_part10_to_part11,area_ratio_part3_to_part9
0,Black_Footed_Albatross_0001_796111.jpg,0,1733.0,2.802498e+02,2.451833,0.396205,0.636197,0.913045,-0.981756,2.772796e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Black_Footed_Albatross_0002_55.jpg,0,3173.0,3.209716e+02,1.939136,0.436571,0.685017,0.856773,-0.954378,3.870319e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Black_Footed_Albatross_0003_796136.jpg,0,1484.0,2.132670e+02,3.039578,0.607699,0.745729,0.944332,1.424403,4.100118e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Black_Footed_Albatross_0005_796090.jpg,0,3035.0,4.076762e+02,1.116284,0.438837,0.622692,0.444399,-0.610094,2.294763e-01,...,0.104427,0.021174,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Black_Footed_Albatross_0006_796065.jpg,0,1874.0,2.030538e+02,2.810191,0.648892,0.876110,0.934544,-1.383102,5.711591e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11783,Common_Yellowthroat_0118_190805.jpg,199,3405.0,3.869899e+02,2.159456,0.415751,0.718809,0.886317,-0.405297,2.857115e-01,...,0.022003,0.013910,NaN,0.126437,0.200000,NaN,NaN,NaN,NaN,NaN
11784,Common_Yellowthroat_0121_190597.jpg,199,3523.0,3.412203e+02,1.904097,0.553670,0.769382,0.850989,-0.319369,3.802355e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11785,Common_Yellowthroat_0122_190570.jpg,199,987.0,1.331604e+02,1.060823,0.559841,0.885996,0.333743,0.650913,6.994822e-01,...,NaN,0.023815,NaN,NaN,1.152709,NaN,NaN,NaN,NaN,NaN
11786,Common_Yellowthroat_0125_190902.jpg,199,386.0,9.101219e+01,1.442437,0.493606,0.863535,0.720677,1.322599,5.855958e-01,...,0.000416,0.021855,NaN,54.749989,1.042857,0.8,41.999992,NaN,0.05814,NaN


In [6]:
train_df[['class_name', 'image']] = train_df['filepath'].str.split('/', expand=True)
val_df[['class_name', 'image']] = val_df['filepath'].str.split('/', expand=True)
test_df[['class_name', 'image']] = test_df['filepath'].str.split('/', expand=True)
train_df = pd.merge(train_df, full_df, on='image', how='left')
val_df = pd.merge(val_df, full_df, on='image', how='left')
test_df = pd.merge(test_df, full_df, on='image', how='left')
assert train_df.notna().any().any(), "Merge failed — features missing!"
train_df

,img_id,filepath,label,is_train,class_name,image,class_id,part1_area,part1_perimeter,part1_aspect_ratio,...,area_ratio_part8_to_total,area_ratio_part9_to_total,area_ratio_part3_to_part11,area_ratio_part4_to_part8,area_ratio_part4_to_part9,area_ratio_part8_to_part10,area_ratio_part9_to_part10,area_ratio_part3_to_part8,area_ratio_part10_to_part11,area_ratio_part3_to_part9
0,4872,084.Red_legged_Kittiwake/Red_Legged_Kittiwake_...,83,0,084.Red_legged_Kittiwake,Red_Legged_Kittiwake_0039_795429.jpg,83,3232.0,522.345215,1.851210,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4017,070.Green_Violetear/Green_Violetear_0041_79564...,69,1,070.Green_Violetear,Green_Violetear_0041_795648.jpg,69,1044.0,226.503571,6.656366,...,0.000542,NaN,NaN,38.999989,NaN,1.499999,NaN,NaN,NaN,NaN
2,1794,032.Mangrove_Cuckoo/Mangrove_Cuckoo_0005_79459...,31,1,032.Mangrove_Cuckoo,Mangrove_Cuckoo_0005_794599.jpg,31,1627.0,211.580734,2.502074,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6522,112.Great_Grey_Shrike/Great_Grey_Shrike_0010_7...,111,0,112.Great_Grey_Shrike,Great_Grey_Shrike_0010_797023.jpg,111,91.0,58.071068,7.359472,...,0.008025,0.006255,NaN,4.544117,5.830189,NaN,NaN,NaN,NaN,NaN
4,2341,041.Scissor_tailed_Flycatcher/Scissor_Tailed_F...,40,0,041.Scissor_tailed_Flycatcher,Scissor_Tailed_Flycatcher_0058_41948.jpg,40,630.0,112.083260,2.494917,...,0.000167,0.006338,NaN,81.999924,2.157895,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8836,2936,051.Horned_Grebe/Horned_Grebe_0064_35015.jpg,50,0,051.Horned_Grebe,Horned_Grebe_0064_35015.jpg,50,2426.0,246.166519,3.112218,...,NaN,NaN,20.488373,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8837,11511,196.House_Wren/House_Wren_0134_188077.jpg,195,1,196.House_Wren,House_Wren_0134_188077.jpg,195,6.0,5.414214,1.732051,...,0.119519,0.000707,NaN,0.431953,72.999962,NaN,NaN,NaN,NaN,NaN
8838,578,011.Rusty_Blackbird/Rusty_Blackbird_0052_7035.jpg,10,0,011.Rusty_Blackbird,Rusty_Blackbird_0052_7035.jpg,10,1292.0,220.965515,2.215001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.421052,NaN
8839,11494,196.House_Wren/House_Wren_0061_187911.jpg,195,0,196.House_Wren,House_Wren_0061_187911.jpg,195,13675.0,532.658936,1.293826,...,NaN,0.006715,NaN,NaN,3.417218,NaN,NaN,NaN,NaN,NaN


In [7]:
val_df

,img_id,filepath,label,is_train,class_name,image,class_id,part1_area,part1_perimeter,part1_aspect_ratio,...,area_ratio_part8_to_total,area_ratio_part9_to_total,area_ratio_part3_to_part11,area_ratio_part4_to_part8,area_ratio_part4_to_part9,area_ratio_part8_to_part10,area_ratio_part9_to_part10,area_ratio_part3_to_part8,area_ratio_part10_to_part11,area_ratio_part3_to_part9
0,4933,085.Horned_Lark/Horned_Lark_0076_73931.jpg,84,1,085.Horned_Lark,Horned_Lark_0076_73931.jpg,84,1924.0,216.823380,2.455318,...,0.020309,0.021760,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4440,077.Tropical_Kingbird/Tropical_Kingbird_0081_6...,76,1,077.Tropical_Kingbird,Tropical_Kingbird_0081_69656.jpg,76,3934.0,340.421356,2.596235,...,0.017598,0.047808,NaN,1.783333,0.656442,2.068965,5.620690,NaN,NaN,NaN
2,8953,153.Philadelphia_Vireo/Philadelphia_Vireo_0001...,152,1,153.Philadelphia_Vireo,Philadelphia_Vireo_0001_156554.jpg,152,7780.0,439.002106,1.754273,...,0.003004,NaN,NaN,8.157895,NaN,0.183575,NaN,NaN,NaN,NaN
3,3170,055.Evening_Grosbeak/Evening_Grosbeak_0025_372...,54,0,055.Evening_Grosbeak,Evening_Grosbeak_0025_37230.jpg,54,2446.0,385.102600,1.915485,...,0.036432,0.000366,NaN,0.043478,4.333332,149.499924,1.499999,NaN,NaN,NaN
4,3522,061.Heermann_Gull/Heermann_Gull_0090_45834.jpg,60,0,061.Heermann_Gull,Heermann_Gull_0090_45834.jpg,60,2803.0,298.285309,1.976887,...,0.019679,0.011585,55.333328,1.000000,1.698630,NaN,NaN,6.693548,NaN,11.369863
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1763,10028,171.Myrtle_Warbler/Myrtle_Warbler_0054_166985.jpg,170,0,171.Myrtle_Warbler,Myrtle_Warbler_0054_166985.jpg,170,2369.0,259.651794,1.358569,...,NaN,0.000602,NaN,NaN,7.799998,NaN,0.029586,NaN,NaN,NaN
1764,7551,129.Song_Sparrow/Song_Sparrow_0036_121679.jpg,128,1,129.Song_Sparrow,Song_Sparrow_0036_121679.jpg,128,5031.0,411.255890,2.624046,...,0.001723,0.032878,NaN,33.923073,1.778226,NaN,NaN,NaN,NaN,NaN
1765,2952,052.Pied_billed_Grebe/Pied_Billed_Grebe_0037_3...,51,0,052.Pied_billed_Grebe,Pied_Billed_Grebe_0037_35598.jpg,51,2522.0,313.628448,2.835750,...,NaN,NaN,813.999207,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1766,3480,060.Glaucous_winged_Gull/Glaucous_Winged_Gull_...,59,0,060.Glaucous_winged_Gull,Glaucous_Winged_Gull_0013_44381.jpg,59,1305.0,307.764496,2.442199,...,0.001572,0.063755,8.879999,18.444443,0.454795,NaN,NaN,24.666664,NaN,0.608219


In [8]:
test_df

,img_id,filepath,label,is_train,class_name,image,class_id,part1_area,part1_perimeter,part1_aspect_ratio,...,area_ratio_part8_to_total,area_ratio_part9_to_total,area_ratio_part3_to_part11,area_ratio_part4_to_part8,area_ratio_part4_to_part9,area_ratio_part8_to_part10,area_ratio_part9_to_part10,area_ratio_part3_to_part8,area_ratio_part10_to_part11,area_ratio_part3_to_part9
0,3503,061.Heermann_Gull/Heermann_Gull_0077_45711.jpg,60,1,061.Heermann_Gull,Heermann_Gull_0077_45711.jpg,60,1.0,1.000000e-06,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8509,145.Elegant_Tern/Elegant_Tern_0044_150946.jpg,144,0,145.Elegant_Tern,Elegant_Tern_0044_150946.jpg,144,5.0,3.828427e+00,8.660254,...,0.000908,NaN,NaN,2.714285,NaN,NaN,NaN,NaN,NaN,NaN
2,8589,147.Least_Tern/Least_Tern_0036_153658.jpg,146,1,147.Least_Tern,Least_Tern_0036_153658.jpg,146,3328.0,5.021676e+02,1.926722,...,NaN,0.032550,NaN,NaN,0.798144,NaN,NaN,NaN,NaN,NaN
3,329,007.Parakeet_Auklet/Parakeet_Auklet_0072_79592...,6,0,007.Parakeet_Auklet,Parakeet_Auklet_0072_795929.jpg,6,4284.0,5.406884e+02,1.065557,...,NaN,0.012581,54.888882,NaN,1.774194,NaN,NaN,NaN,NaN,7.967742
4,10948,186.Cedar_Waxwing/Cedar_Waxwing_0092_179123.jpg,185,1,186.Cedar_Waxwing,Cedar_Waxwing_0092_179123.jpg,185,5402.0,4.246762e+02,2.077280,...,NaN,0.004979,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1174,8056,138.Tree_Swallow/Tree_Swallow_0092_136236.jpg,137,0,138.Tree_Swallow,Tree_Swallow_0092_136236.jpg,137,263.0,1.140122e+02,5.686695,...,0.000624,NaN,NaN,41.333321,NaN,0.033333,NaN,NaN,NaN,NaN
1175,3156,055.Evening_Grosbeak/Evening_Grosbeak_0078_380...,54,0,055.Evening_Grosbeak,Evening_Grosbeak_0078_38051.jpg,54,298.0,8.456854e+01,2.982295,...,0.001571,0.008528,NaN,NaN,NaN,NaN,NaN,34.714279,NaN,6.394737
1176,6241,107.Common_Raven/Common_Raven_0128_102017.jpg,106,0,107.Common_Raven,Common_Raven_0128_102017.jpg,106,7672.0,7.463290e+02,3.086730,...,0.008405,0.006680,NaN,2.113537,2.659341,NaN,NaN,NaN,NaN,NaN
1177,5636,097.Orchard_Oriole/Orchard_Oriole_0042_91678.jpg,96,1,097.Orchard_Oriole,Orchard_Oriole_0042_91678.jpg,96,28.0,1.848528e+01,1.902564,...,0.069296,NaN,NaN,0.063830,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
print(f"Shape of dataset, train: {train_df.shape}, val: {val_df.shape}, test: {test_df.shape}")

Shape of dataset, train: (8841, 2869), val: (1768, 2869), test: (1179, 2869)


In [10]:
train_df = fill_by_class_mean(train_df)
val_df   = fill_by_class_mean(val_df)
test_df  = fill_by_class_mean(test_df)

In [11]:
feature_cols = train_df.drop(columns=columns_to_drop.tolist() + ["image", "class_id", "class_name"]).columns

X_train = train_df[feature_cols].values
X_val   = val_df[feature_cols].values
X_test  = test_df[feature_cols].values

y_train = train_df["label"].values
y_val   = val_df["label"].values
y_test  = test_df["label"].values

# Standardize
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_val_std   = scaler.transform(X_val)
X_test_std  = scaler.transform(X_test)

classes = np.unique(y_train)

# --- Sanity check ---
assert not np.isnan(X_train_std).any(), "NaNs remain in X_train_std!"
assert not np.isnan(X_val_std).any(), "NaNs remain in X_val_std!"
assert not np.isnan(X_test_std).any(), "NaNs remain in X_test_std!"
print("✅ Class-wise NaN imputation complete — no missing values remain.")

✅ Class-wise NaN imputation complete — no missing values remain.


In [12]:
# Fit PCA with all components
pca_full = PCA(n_components=X_train_std.shape[1], random_state=SEED)
pca_full.fit(X_train_std)

# Cumulative explained variance
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# Find number of components to explain desired variance (e.g., 95%)
explained_threshold = EXPLAINED_VARIANCE_THRESHOLD
n_components_opt = np.argmax(cumulative_variance >= explained_threshold) + 1

print(f"Optimal n_components for {explained_threshold*100}% variance: {n_components_opt}")

Optimal n_components for 99.0% variance: 1989


In [13]:
methods = ["PCA", "ICA", "TruncatedSVD", "GRP", "SRP", "NMF"]
best_acc = -1
best_method = None
best_data = None

device = torch.device(f"cuda:{DEVICE_ID}")

summary_rows = []

for method in methods:
    print(f"\n================ {method} ================\n")

    Xtr, Xva, Xte, reducer = reduce_dimensions(
        X_train_std, X_val_std, X_test_std,
        method=method,
        n_components=n_components_opt
    )

    train_dataset = ShapeFeatureDataset(Xtr, y_train)
    val_dataset   = ShapeFeatureDataset(Xva, y_val)
    test_dataset  = ShapeFeatureDataset(Xte, y_test)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    model = MLPClassifier(Xtr.shape[1], len(classes), dropout=DROPOUT).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)

    history = train_model(
        model, train_loader, val_loader,
        criterion, optimizer, scheduler,
        device, EPOCHS,
        LOG_DIR + f"_{method}"
    )

    # ---- load best checkpoint ----
    best_path = os.path.join(LOG_DIR + f"_{method}", "checkpoints", "best_model.pth")
    model.load_state_dict(torch.load(best_path, map_location=device))

    val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)
    test_loss, test_acc = validate_one_epoch(model, test_loader, criterion, device)

    print(f"{method} Val Acc: {val_acc:.4f} | Test Acc: {test_acc:.4f}")

    summary_rows.append({
        "method": method,
        "original_dim": X_train_std.shape[1],
        "reduced_dim": Xtr.shape[1],
        "val_acc": val_acc,
        "test_acc": test_acc
    })

    if test_acc > best_acc:
        best_acc = test_acc
        best_method = method
        best_data = (Xtr, Xva, Xte)


================ PCA ================

Starting the first epoch...
 Epoch 1/100 | Train Loss: 4.1031 | Train Acc: 0.3111 | Val Loss: 4.8442 | Val Acc: 0.0560
 Epoch 2/100 | Train Loss: 1.3074 | Train Acc: 0.9677 | Val Loss: 4.6608 | Val Acc: 0.0877
 Epoch 3/100 | Train Loss: 1.0501 | Train Acc: 0.9982 | Val Loss: 4.5973 | Val Acc: 0.0922
 Epoch 4/100 | Train Loss: 1.0088 | Train Acc: 1.0000 | Val Loss: 4.5038 | Val Acc: 0.1029
 Epoch 5/100 | Train Loss: 0.9769 | Train Acc: 1.0000 | Val Loss: 4.4969 | Val Acc: 0.1052
 Epoch 6/100 | Train Loss: 0.9615 | Train Acc: 1.0000 | Val Loss: 4.5128 | Val Acc: 0.1052
 Epoch 7/100 | Train Loss: 0.9537 | Train Acc: 1.0000 | Val Loss: 4.5243 | Val Acc: 0.1024
 Epoch 8/100 | Train Loss: 0.9480 | Train Acc: 1.0000 | Val Loss: 4.5345 | Val Acc: 0.1058
 Epoch 9/100 | Train Loss: 0.9437 | Train Acc: 1.0000 | Val Loss: 4.5421 | Val Acc: 0.1029
 Epoch 10/100 | Train Loss: 0.9399 | Train Acc: 1.0000 | Val Loss: 4.5482 | Val Acc: 0.1058
 Epoch 11/100 | Train

In [14]:
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(LOG_DIR+"cub_dim_reduction_summary.csv", index=False)

print("\nSaved summary to dim_reduction_summary.csv")
print(summary_df)


Saved summary to dim_reduction_summary.csv
         method  original_dim  reduced_dim   val_acc  test_acc
0           PCA          2862         1989  0.105204  0.114504
1           ICA          2862         1989  0.073529  0.063613
2  TruncatedSVD          2862         1989  0.113122  0.091603
3           GRP          2862         1989  0.144796  0.122986
4           SRP          2862         1989  0.136878  0.116200
5           NMF          2862         1989  0.018100  0.005937


In [15]:
print(f"\n🏆 Best method: {best_method} (Test Acc={best_acc:.4f})")

Xtr_best, Xva_best, Xte_best = best_data

cols = [f"{best_method}_comp_{i}" for i in range(Xtr_best.shape[1])]

train_out = pd.DataFrame(Xtr_best, columns=cols)
train_out["label"] = y_train

val_out = pd.DataFrame(Xva_best, columns=cols)
val_out["label"] = y_val

test_out = pd.DataFrame(Xte_best, columns=cols)
test_out["label"] = y_test

train_out.to_csv(LOG_DIR+f"cub_reduced_train_{best_method}.csv", index=False)
val_out.to_csv(LOG_DIR+f"cub_reduced_val_{best_method}.csv", index=False)
test_out.to_csv(LOG_DIR+f"cub_reduced_test_{best_method}.csv", index=False)

print("✅ Saved reduced datasets for best method.")


🏆 Best method: GRP (Test Acc=0.1230)
✅ Saved reduced datasets for best method.
